# Strawberry Data Audit

Use this notebook before changing model configs. It confirms split health, label ranges, environment ranges, and sequence counts.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

LAB_DIR = Path.cwd()
if LAB_DIR.name != 'strawberry':
    LAB_DIR = Path('notebooks/strawberry').resolve()
sys.path.insert(0, str(LAB_DIR))
import lab_utils as lab


In [ ]:
labels = lab.load_all_labels()
print(labels.shape)
labels[['split', 'fruit_id', 'timestamp', 'rul_hours', 'temperature_c', 'humidity_pct']].head()


In [ ]:
summary = labels.groupby(['split', 'fruit_id']).agg(
    frames=('rul_hours', 'size'),
    rul_min=('rul_hours', 'min'),
    rul_mean=('rul_hours', 'mean'),
    rul_median=('rul_hours', 'median'),
    rul_max=('rul_hours', 'max'),
    pct_zero=('rul_hours', lambda s: (s == 0).mean()),
    temp_min=('temperature_c', 'min'),
    temp_max=('temperature_c', 'max'),
    hum_min=('humidity_pct', 'min'),
    hum_max=('humidity_pct', 'max'),
).reset_index()
summary


In [ ]:
labels[['rul_hours', 'temperature_c', 'humidity_pct']].corr(numeric_only=True)


In [ ]:
counts = lab.sequence_counts([3, 5, 8, 10])
counts.groupby(['split', 'seq_len'])['sequences'].sum().unstack('seq_len')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels['rul_hours'].hist(ax=axes[0], bins=40)
axes[0].set_title('RUL distribution')
labels['temperature_c'].hist(ax=axes[1], bins=20)
axes[1].set_title('Temperature')
labels['humidity_pct'].hist(ax=axes[2], bins=20)
axes[2].set_title('Humidity')
plt.tight_layout()


## Notes

The environment variables are real inputs, but in this dataset their global linear correlation with RUL is weak. Treat them as side/context features rather than the primary degradation signal.
